The only thing we see is that at layers.2, out12, in6, (2,4), we always have a `| |` alignment, which is detecting the pattern of empty space between two diagonal `/` lines.   

Sometimes it is spurious, but this suggests that there is high spatial correlation in the data about this point.  
We might see more examples of these points in this kernel around this space.  
Maybe in the area from [2:4, 3:5], I can check more points around that space to see if we find more.   
It might be interesting to actually see how this slice came about though.  

I should write down the propositions for one of them.  
Somehow.  


# Proposition chain
For the URL:
http://localhost:5173/models/simple_mnist_v1/4-oVe/v1/kernel-slice/layers.2/12/6


- Input pixels has some black parts and some green parts.  
- Its best to not write propositions spatially as it might hurt my judgement, I dont really want to see what the set of propositions looks like in my head.  

- Each pixel carries a proposition, the proposition talks about how its patch was aligned for now.  

Main POI: `{layers.2, out.12, in.6}[2,4]`
Receptive field: `{layers.1, out.6}[3:6, 7:10]`
Receptive field: `{layers.0, out.6}[3:6, 7:10]`

Receptive field of the layers.0 patch (only one input channel haash)
The final receptive field: `{x, out.0}[5:11, 13:19]`


## {layers.0} propositions

Two labels: `all red`, `some green`   
Two output labels: `green`, `red`

```js
// the source vs target pattern, this is also important
[3,7]: (all red -> green)
[3,8]: (all red -> green)
[3,9]: (all red -> green)
[4,7]: (some green -> red)
[4,8]: (all red -> green)
[4,9]: (some green -> red)
[5,7]: (some green -> red)
[5,8]: (all red -> green)
[5,9]: (some green -> red)
```

We can write the full pixel as `{layers.0, out.6}[3,7](all red -> green)`

## {layers.1} propositions
```js
// the source vs target pattern, this is also important
[3,7]: (green -> green)
[3,8]: (green -> green)
[3,9]: (green -> green)
[4,7]: (red -> black)
[4,8]: (green -> green)
[4,9]: (red -> black)
[5,7]: (red -> black)
[5,8]: (green -> green)
[5,9]: (red -> black)
```
This is a pointwise operation only, for our case, the propositions dont change much.  
They do see some destruction in general from ReLU though (all red information is empty for the network, other than patterns though, basically the network can't reason about the reds from magnitude, it can use spatial patterns though).  
For this case, it does not matter much.   

It would later be useful to see these as 1d arrays, so that i can see the propositions without seeing the image.  

## {layers.2} propositions


Now the interesting part is that there is some pattern the kernel is using to determine the uniqueness of the output given by the final layer.  
This is the first question, can the final layer actually differentiate between the multiple patterns defined by a kernel?  
It might be useful to see the weight corresponding to this (the flattened weight of the last layer, unflattens to our channel, lets see (although, it is dependent on the cube and not the slice that im thinking right now)).  

Let us assume that we cannot differentiate from the input.  
In this case, the final pattern actually is forming a diagonal, from the kernels final state, i can only say that the green color means:

- `/` or a `| |` (there is however, a high correlation of finding `| |` at (2,4) lol, is this a spatial thing?).  
- The final weight is really just a multiplication, lets see how it goes.  

For this kernel however, there is only:
- perfect alignment which -> `| |`
- or `/` for some range

Lets verify this.  

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
import sys
import django

# Setup Django environment
# Adjust the path to point to the directory containing manage.py
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../hiccup_ide")))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hiccup_ide.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

In [ ]:
from neural_data.models import *
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import show_single_channel_red_green_black as S, to_show_list as tsl
from pt_to_api.contribs.v1 import show_input_patch_and_kernel_placement_for_poi_using_raw_params as SIP

In [ ]:
def _get_acts_of_sm(sm):
    act_sm = Activation.objects.get(coordinate=sm.coordinate, input=sm.input)
    act_0 = Activation.objects.get(coordinate="layers.0.out_6", input=sm.input)
    act_x = Activation.objects.get(coordinate="x.out_0", input=sm.input)
    return act_x, act_0, act_sm


def show_patches(samples, predicate, position, num_samples=10, print_meta=True):
    (y,x) = position
    subsamples = [a for a in samples if predicate(a)]
    print("total length", len(subsamples)) # very few
    subsamples = random.sample(subsamples, min(num_samples, len(subsamples)))
    for sample in subsamples:
        act = np.array(sample[-2].data)
        print("contrib", sample[0])
        SIP((3,3), (2,2), (1,1), kernel, act, y, x, "dark", print_meta=print_meta)
        plt.show()


kernel = np.array(Weight.objects.get(coordinate='layers.2.out_12.in_6', model__alias='simple_mnist_v1').data)

# Verify (2,4)'s range

In [ ]:
sms = SaliencyMap.objects.filter(coordinate="layers.2.out_12.in_6")
sms_2_4 = [sm for sm in sms if sm.data[2][4] > 0]
sms.count(), len(sms_2_4)

In [ ]:

def _get_acts_of_sm(sm):
    act_sm = Activation.objects.get(coordinate=sm.coordinate, input=sm.input)
    act_0 = Activation.objects.get(coordinate="layers.0.out_6", input=sm.input)
    act_x = Activation.objects.get(coordinate="x.out_0", input=sm.input)
    return act_x, act_0, act_sm

val_sm_acts = [(sm.data[2][4], sm, *_get_acts_of_sm(sm)) for sm in sms_2_4]
kernel = np.array(Weight.objects.get(coordinate='layers.2.out_12.in_6', model__alias='simple_mnist_v1').data)
sm = sms_2_4[0]
act_x, act_0, act_sm = _get_acts_of_sm(sm)

print(len(val_sm_acts))
S([
    np.array(act_x.data), 
    np.array(act_0.data),
    np.array(act_sm.data),
], 10, ncols=3, viztype="local")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming your data is named 'data_list'
# data_list = [(10.5, array1), (11.2, array2), (25.0, array3), ...]

# 1. Extract just the numbers (the first element of each tuple)
numbers = [item[0] for item in val_sm_acts]

# 2. Create the visualization
plt.figure(figsize=(10, 2))
sns.stripplot(x=numbers, color='blue', alpha=0.5, jitter=True)

plt.title('1D Clustering Visualization')
plt.xlabel('Value')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# there are a lot near the 0s, lets check them out
# lets see the ones < 0.001
less_vals = [a for a in val_sm_acts if a[0] < 0.001]
# we need the activation first for them
S([np.array(a[2].data) for a in less_vals[:4]], 10, ncols=4)
S([np.array(a[3].data) for a in less_vals[:4]], 10, ncols=4)
S([np.array(a[4].data) for a in less_vals[:4]], 10, ncols=4)

In [ ]:
# i had forgotten to use only 4s, but eh anyways
samples = random.sample(less_vals, 10)
for sample in samples:
    act = np.array(sample[-2].data)
    SIP((3,3), (2,2), (1,1), kernel, act, 2, 4, "dark", print_meta=False)

In [ ]:
# lets do 0.001 -> 0.003
# these do have some semblence of attaching here
# the alignment is generally not strong, but not too weak
# it is useful to see the output here too i feel
vals_01_to_03 = [a for a in val_sm_acts if 0.001 < a[0] <= 0.003]
print("total length", len(vals_01_to_03)) # very few
samples = vals_01_to_03
for sample in samples:
    act = np.array(sample[-2].data)
    SIP((3,3), (2,2), (1,1), kernel, act, 2, 4, "dark", print_meta=False)

In [ ]:
# we do see diagonal patterns coming up from time to time
# in the output, a / coming in output does suggest that there is a diagonal representation in input
# that does cut the assertion that its a | | pattern here
# is it what the network is using? i dont know, for now, no need since in convs, we know
# that they look at spatial positions (I would need to in end give up on linear layers for now)
S([np.array(a[-2].data) for a in samples], 20, ncols=5)
plt.show()
S([np.array(a[-1].data) for a in samples], 20, ncols=5)
plt.show()

In [ ]:
# lets look at the other groups now
vals_gt_03 = [a for a in val_sm_acts if 0.003 < a[0]]
print("total length", len(vals_gt_03)) # very few
samples = vals_gt_03
for sample in samples:
    act = np.array(sample[-2].data)
    SIP((3,3), (2,2), (1,1), kernel, act, 2, 4, "dark", print_meta=False)

It seems the major push comes from the left side alone. It is basically finding a diagonal on the left side then only, the push from the right side is not that strong.  

So it seems I've come across a very simple kernel, which is like a binary switch for 4.  
Now, it would be important to see how (2,4) as a POI acts for other samples. I dont expect it to work well for others. It works because the receptive field is large, but for me to actually make sense of it, i need to make sense of the propositions.  
Do note that high negs on the right side might actually give good results, and (2,4) is not the only candidate in this kernel.  

Now lets look at all the others in this spatial kernel.   
We will take many samples in different ranges, I've got a reasonable idea now though.  

# All POIs in the slice

In [ ]:
def get_pos(sm):
    res = []
    for r, row in enumerate(sm.data):
        for c, col in enumerate(row):
            if sm.data[r][c] > 0:
                res.append((col, (r, c), sm))
    return res

In [ ]:
import itertools
sms = SaliencyMap.objects.filter(coordinate="layers.2.out_12.in_6")
sms_all_pois = [get_pos(sm) for sm in sms]
sms_all_pois = [a for a in sms_all_pois if len(a) > 0]
sms_all_pois = list(itertools.chain.from_iterable(sms_all_pois))
sms.count(), len(sms_all_pois)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def _scatter_plot_1d(numbers, suff=""):

    # 2. Create the visualization
    plt.figure(figsize=(10, 2))
    sns.stripplot(x=numbers, color='blue', alpha=0.5, jitter=True)

    plt.title('1D Clustering Visualization' + suff)
    plt.xlabel('Value')
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.show()

In [ ]:
_scatter_plot_1d([item[0] for item in sms_all_pois])

hmmmm hmmmm hmmmm, the clustering is interesting, not the easiest to look at but oh well.  
Lets try first grouping by coords

In [ ]:
from collections import defaultdict
coord_by_pois = defaultdict(list)
for item in sms_all_pois:
    coord_by_pois[item[1]].append((item[0], item[2], *_get_acts_of_sm(item[2])))

## row=0

These are basically the ones where the padding also comes in, its hard to show it so im going to skip it.  

In [ ]:
# these all are quite quite low l
# there is no usefulness here
# all the eles in first row seem to be this
# very less contribs
# it might make sense to prune them a bit but its okay
for r in [0]:
    for c in range(6):
        coord = (r,c)
        _scatter_plot_1d([item[0] for item in coord_by_pois[coord]], str(coord))

## row=1

In [ ]:

for r in [1]:
    for c in range(6):
        coord = (r,c)
        _scatter_plot_1d([item[0] for item in coord_by_pois[coord]], str(coord))

In [ ]:
# (1,4) and (1,5) have some outliers, lets check them out
# lets look at the other groups now
# vertical edge of sorts detection
y, x = 1, 4
samples = coord_by_pois[y,x]
predicate = lambda a: a[0] > 0.001

subsamples = [a for a in samples if predicate(a)]
print("total length", len(subsamples)) # very few
for sample in subsamples:
    act = np.array(sample[-2].data)
    SIP((3,3), (2,2), (1,1), kernel, act, y, x, "dark", print_meta=False)

In [ ]:
# (1,4) and (1,5) have some outliers, lets check them out
# lets look at the other groups now
# vertical edge of sorts detection
# similar, but rare, very weird that the model is using these, but again, rare
# statistically these are not the significant points of this kernel
# i understand now, looking at one point input at a time is not very useful
# i need to see the statistical pattern across all inputs
y, x = 1, 5
samples = coord_by_pois[y,x]
predicate = lambda a: a[0] > 0.001

subsamples = [a for a in samples if predicate(a)]
print("total length", len(subsamples)) # very few
for sample in subsamples:
    act = np.array(sample[-2].data)
    SIP((3,3), (2,2), (1,1), kernel, act, y, x, "dark", print_meta=False)

## row=2

In [ ]:
for r in [2]:
    for c in range(6):
        coord = (r,c)
        _scatter_plot_1d([item[0] for item in coord_by_pois[coord]], str(coord))

(2,3) (2,4) and (2,5) show useful patterns.  
Now I've to remember that im only doing point wise clustering, later i would need to cluster for patterns too since downstream kernels rely on that.  
For now, we are only relying on contribs calculation to give us maximally active calculations  
Although, i should still see if there is a correlation with activation output, but a contrib might be able to give me good correlation because it factors in the pattern the downstream layers use.  

It might be useful to find the best scheme which gives us useful clustering of kernel activations based on the assigned contrib alone. For this I will need to change how the calculation is done (currently, it only cares about the positive values, i dont think that is enough, we need to model the absence also).  

For now, I already see that contrib < 0.001 is quite bad, i'll confirm it with (2,3) for now

### (2,3)

Passes through the left stem basically

In [ ]:
coord = (2, 3)
_scatter_plot_1d([item[0] for item in coord_by_pois[coord]], str(coord))

In [ ]:
# not good, generally misalignemnts only, none of the activations are particularly high too
show_patches(coord_by_pois[coord], lambda a: a[0] < 0.001, coord, 5)

In [ ]:
# we have some vertical edges starting now
show_patches(coord_by_pois[coord], lambda a: 0.002 < a[0] < 0.004, coord, 10)

In [ ]:
# this is the same as above
show_patches(coord_by_pois[coord], lambda a: 0.004 < a[0] < 0.005, coord, 10)

In [ ]:
show_patches(coord_by_pois[coord], lambda a: 0.005 < a[0] < 0.008, coord, 10)

In [ ]:
show_patches(coord_by_pois[coord], lambda a: 0.008 < a[0], coord, 10)

We start seeing vertical edges now, quite interesting that the contribution assigned to these POIs is larger than the contribution assigned to (2,4).   
(2,4) had good alignments (very high activations) and it still got lesser priority than (2,3) which has vertical alignments at its max contribs, which reach upto 0.012 (while [2,4] reached upto 0.007).  
In the end, i do need to look at the flattened weight. I don't expect the LRP to be wrong here though, i'll confirm by looking at the weights to see if more has been assigned to (2,3) (i expect its weight to be higher right now, assuming that input activation density is constant).  

I do now believe that i feel we need a better contrib calculating scheme for conv kernels, one which is more geared towards their behavior, it might be the answer of getting good clustering based on contributions itself.  

The clustering hasn't told us something useful other than the fact that killing contribs below 0.001 is giving consistently good pattern recognition in this kernel.  
I'll have to think more about the point "if the network decided to use it, its clusterable" philosophy.  

## (2,4)

Higher contribs (and even the range 0.002 -> 0.003) consistently is trying to align with the right and left stem both. Maybe that is a booster, im not sure.   
The right side of the kernel has weaker red, so im again assuming that it is a booster.  

There is not enough information for me to verify on whether the kernel is actually aligning perfectly or not. Looking at the evidence, it still seems that any of `/` or `| |` are fine with the network.  

In [ ]:
# quite low density
coord = (2, 4)
_scatter_plot_1d([item[0] for item in coord_by_pois[coord]], str(coord))

In [ ]:
# what is interesting is that no alignment is sometimes given a slightly positive contrib
# the result is negative, we have this slightly positive contribs for 4 also, ill have to check this
show_patches(coord_by_pois[coord], lambda a: a[0] < 0.001, coord, 10, print_meta=True)

In [ ]:
show_patches(coord_by_pois[coord], lambda a: 0.001 < a[0] < 0.003, coord, 10, print_meta=True)

In [ ]:
show_patches(coord_by_pois[coord], lambda a: 0.003 < a[0] , coord, 10, print_meta=True)

## (2,5)

passes through the right stem.  

In [ ]:
coord = (2, 5)
_scatter_plot_1d([item[0] for item in coord_by_pois[coord]], str(coord))

In [ ]:
# interesting that negative activations are saying "yes" to 4 when its not that class lol
# we have positive contribs for us from this class it seems generally (the activation value really are negative though lol)
# sometimes positive too, quite weird.  
# havent added bias though
show_patches(coord_by_pois[coord], lambda a: a[0] < 0.001, coord, 10, print_meta=True)

In [ ]:
# well this is quite random i feel, why do we have low contrib?
# we ll also need to look at the main channel to see if we are killed btw yes
# or if someone else is stronger, there can be many reasons
show_patches(coord_by_pois[coord], lambda a: 0.001 < a[0] < 0.003, coord, 10, print_meta=True)

In [ ]:
# well this is quite random i feel, why do we have low contrib?
# we ll also need to look at the main channel to see if we are killed btw yes
# or if someone else is stronger, there can be many reasons
show_patches(coord_by_pois[coord], lambda a: 0.003 < a[0] < 0.005, coord, 10, print_meta=True)

In [ ]:

# well this is quite random i feel, why do we have low contrib?
# we ll also need to look at the main channel to see if we are killed btw yes
# or if someone else is stronger, there can be many reasons
show_patches(coord_by_pois[coord], lambda a: 0.005 < a[0] < 0.008, coord, 10, print_meta=True)

In [ ]:
# this is quite sus now, what is happening lol
# this kernel is giving negative values, but has a positive contribution
# that generally means that the out contrib itself should have a negative contribution to 4
# this might be the case, let me see
show_patches(coord_by_pois[coord], lambda a: 0.008 < a[0] , coord, 10, print_meta=True)

# (2,6)

In [ ]:
coord = (2, 6)
_scatter_plot_1d([item[0] for item in coord_by_pois[coord]], str(coord))

In [ ]:
# well pretty bad, does have a pattern though, solid green input patch lol
show_patches(coord_by_pois[coord], lambda a: a[0] < 0.0015, coord, 10)

In [ ]:
show_patches(coord_by_pois[coord], lambda a: 0.0015 < a[0] < 0.002, coord, 10)

In [ ]:
# damping of the bright reds has started
# generally towards to the top part of the edge
show_patches(coord_by_pois[coord], lambda a: 0.002 < a[0] < 0.003, coord, 10)

In [ ]:
# starting of some damping of the bright reds in the kernel
# no significant difference from the previous band, its basically saying 
# "there is something vertical here"
# how these compose with the previous propositions for the full receptive field is yet to be seen
show_patches(coord_by_pois[coord], lambda a: 0.003 < a[0], coord, 10)

# Row 3

There are some outliers but the whole row is quite useless. Skipping.  

In [ ]:
r = 3
for c in range(6):
    coord = (r,c)
    _scatter_plot_1d([item[0] for item in coord_by_pois[coord]], str(coord))

# Row 4
Same as row 3

In [ ]:
r = 4
for c in range(6):
    coord = (r,c)
    _scatter_plot_1d([item[0] for item in coord_by_pois[coord]], str(coord))

# Row 5
We good, i dont see a lot of contribs here.  

In [ ]:
r = 5
for c in range(6):
    coord = (r,c)
    _scatter_plot_1d([item[0] for item in coord_by_pois[coord]], str(coord))

# Row 6

Useless

In [ ]:
r = 6
for c in range(6):
    coord = (r,c)
    _scatter_plot_1d([item[0] for item in coord_by_pois[coord]], str(coord))